# Data Pre-Processing for Asl-Sign-50

This is the pre-processing notebook for cleaning and normalizing the media-piped 50 unique sign data from the: Google - Isolated Sign Language Recognition Dataset

## Data Discovery and Exploration

In [6]:
import pandas as pd
import numpy as np
import os

print("All imports successful")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

All imports successful
NumPy: 2.2.6
Pandas: 2.3.3


In [7]:
# Configuration
DATA_DIR      = './asl-signs-50'
TRAIN_CSV     = os.path.join(DATA_DIR, 'train.csv')
OUTPUT_DIR    = './data/landmarks' # output directory to save landmarks
TARGET_FRAMES = 30
MIN_FRAMES    = 5
RANDOM_SEED  = 69

# verify paths exist
print(f"Data dir exists: {os.path.exists(DATA_DIR)}")
print(f"Train CSV exists: {os.path.exists(TRAIN_CSV)}")

# load and preview train.csv
train = pd.read_csv(TRAIN_CSV) # csv that holds the "Table of contents" of the data
print(f"\nTotal samples: {len(train)}")
print(f"Columns: {train.columns.tolist()}")
print(f"\n=== First 5 Rows ===")
print(train.head())
print("\n=== Statistical Summary ===")
print(train.describe())
print("\n=== Column Data Types ===")
print(train.dtypes)

Data dir exists: True
Train CSV exists: True

Total samples: 18907
Columns: ['path', 'participant_id', 'sequence_id', 'sign']

=== First 5 Rows ===
                                            path  participant_id  sequence_id  \
0  train_landmark_files/26734/1000035562.parquet           26734   1000035562   
1  train_landmark_files/28656/1000106739.parquet           28656   1000106739   
2  train_landmark_files/25571/1000210073.parquet           25571   1000210073   
3  train_landmark_files/26734/1000241583.parquet           26734   1000241583   
4  train_landmark_files/27610/1000956928.parquet           27610   1000956928   

   sign  
0  blow  
1  wait  
2  bird  
3  duck  
4   owl  

=== Statistical Summary ===
       participant_id   sequence_id
count    18907.000000  1.890700e+04
mean     33756.297879  2.149644e+09
std      16052.215389  1.231101e+09
min       2044.000000  1.649177e+06
25%      25571.000000  1.094789e+09
50%      32319.000000  2.140024e+09
75%      49445.000000  3

### Data set information
**Train.csv:** is a look up table that maps each video clip to its label, it is a csv of pointers (table of contents for the data).
path -> where is the data file
participant_id -> who signed it  
sequence_id -> unique clip ID
sign -> what word they signed

**Train_Landmark_files/:** This is where the actual data lives with all of the media piped coordinates for each frame of each sign for each sample. <br>
├── 26734/                  //participant ID (one of 21 signers) <br>
│   ├── 1000035562.parquet  //one clip (person signing "a sign name (ex: wolf) here") <br>

**Statistical Summary:**
The count shows that there are no missing values in either numeric column.


In [8]:
# Print all of the different signs
print("\nAll signs and sample counts:") # print the unique signs and how many samples of each sign we have in the training set
print(train['sign'].value_counts().to_string())  # prints the amount each value is in training data
print(f"\nMean Samples for each sign: {train.groupby('sign').size().mean()}")
print(f"\nTotal unique signs: {train['sign'].nunique()}") # print the total number of unique signs in the training set


All signs and sample counts:
sign
duck         405
bird         404
cow          404
drink        400
cat          400
make         398
find         397
owl          396
pizza        396
frog         396
bee          395
goose        394
up           394
tiger        394
airplane     393
lion         392
blow         391
horse        391
cry          390
alligator    390
finish       388
wolf         388
snow         386
yes          386
jump         383
fall         382
rain         381
same         380
dog          380
fish         380
later        377
close        374
boat         370
no           370
now          369
cut          369
pig          368
every        368
hide         363
read         362
fast         362
quiet        358
drop         356
any          355
open         354
ride         347
wait         346
give         346
down         327
dance        312

Mean Samples for each sign: 378.14

Total unique signs: 50


**Initial 50 Signed training sample counts**
There is a decent representation for each sign. The samples for each sign range from 312-405.

In [10]:
# load one parquet file and inspect it
sample_path = os.path.join(DATA_DIR, train['path'][0])
df = pd.read_parquet(sample_path)

print(f"Shape: {df.shape}") # print the shape of the data frame, each row is one landmark for one frame 
print(f"\nFirst few columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows, first 10 columns:")
print(df.iloc[:3, :])
print(f"\nNaN count per frame (first 5 frames):")
print(df.iloc[:5].isna().sum().sum())


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/cre37/.local/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/cre37/.local/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/cre37/.local/lib/python3.10/site-packag

AttributeError: _ARRAY_API not found

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

**Data Format:**
The data is in long format, each row is one landmark per frame.

We have to restructure the data to define individual samples more clearly for training.

In [ ]:
# Looking at a sample before interpolating the data 
print(f"Unique types: {df['type'].unique()}")
print(f"Unique frames: {df['frame'].nunique()}")
print(f"Rows per frame: {len(df) / df['frame'].nunique():.0f}")

# look at just hand rows
hands = df[df['type'].isin(['left_hand', 'right_hand'])]
print(f"\nHand rows: {len(hands)}")
print(f"Hand types: {hands['type'].unique()}")
print(f"Landmark indices: {hands['landmark_index'].unique()}")

**Data Description:**

Landmark types: face, pose, left_hand, right_hand —> we know exactly what to filter; we only want hands <br>
Confirms that there are 543 landmarks per frame <br>
Confirms that there are 21 landmarks per hand, 21*(2 hands)*(3 dimensions x,y,z) = 126 landmark features per frame. <br>

## Data Cleaning 

**Data Leakage:** There is no initial target leakage, and we will normalize and separate data early to prevent train-test contamination

**Missing Values:** There will be two sources of missing data: frames where no hand was detected and actual instances where one of the hand landmarks are zero. We will handle no detections of hands by replacing NaN with 0 and dropping all of the zero frames. This will tell the model that a hand is not present which will be correct for the interpretation of the data, as the zeros carry meaning. If there are specific landmarks within a hand missing, that is very rare so the replacement with zero will still work. 

**Categorial Features:** The label sign classes is a categorical output and we will use ordinal encoding (integers 0-49) made through a LabelEncoder just for lables, it is not adding any meaning or relationships based on the labels. 

**Approach:**
Load only x, y, z columns (this will be faster and simpler)
Reshape assuming exactly 543 rows per frame into (n_frames, 543, 3)
Extract hand landmark indices directly by position since the order is always (as stated by the competition that provided the dataset (https://www.kaggle.com/competitions/asl-signs/overview/evaluation):
    Face: indices 0-467 (468 landmarks)
    Pose: indices 468-500 (33 landmarks)
    Left hand: indices 501-521 (21 landmarks)
    Right hand: indices 522-542 (21 landmarks)

In [ ]:
# The dataset has exactly 543 landmarks per frame:
# face: 0-467, pose: 468-500, left_hand: 501-521, right_hand: 522-542
ROWS_PER_FRAME   = 543
LEFT_HAND_IDX    = slice(501, 522)  # 21 landmarks
RIGHT_HAND_IDX   = slice(522, 543)  # 21 landmarks

# Wrist is landmark index 0 within each hand
# In our final 126-feature vector layout:
#  left x,y,z 0-20                   right x,y,z 0-20
# [lx0..lx20, ly0..ly20, lz0..lz20, rx0..rx20, ry0..ry20, rz0..rz20]
# Left wrist:  feature index 0 (x), 21 (y), 42 (z)
# Right wrist: feature index 63 (x), 84 (y), 105 (z)

# ── load_parquet function ─────────────────────────────────────────────────────
# Takes a path to one parquet file (one signing clip)
# Returns a cleaned (T, 126) numpy array, or None if the clip is unusable
def load_parquet(path):
    
    # load only the x, y, z coordinate columns from the parquet file
    df = pd.read_parquet(path, columns=['x', 'y', 'z'])
    
    # calculate how many frames
    # total rows / 543 landmarks per frame = number of frames
    n_frames = int(len(df) / ROWS_PER_FRAME)
    
    # skip this clip if it has no frames 
    if n_frames == 0:
        return None
    
    # reshape (n_frames*543, 3) array into (n_frames, 543, 3)
    # so we can index by [frame, landmark_index, coordinate]
    # example: data[5, 501, 0] = x coordinate of left wrist in frame 5
    data = df.values.reshape(n_frames, ROWS_PER_FRAME, 3).astype(np.float32) #reshapes the 2d array of x,y,z landmarks into 3D array of frame, landmark, coordinates
    
    # extract only the hand landmarks from the full 543-landmark array based off of the slice
    # left hand:  landmarks 501-521 → shape (n_frames, 21, 3)
    # right hand: landmarks 522-542 → shape (n_frames, 21, 3)
    left  = data[:, LEFT_HAND_IDX,  :]
    right = data[:, RIGHT_HAND_IDX, :]
    
    # build our 126-feature vector for each frame by concatenating:
    # all left hand x coords, then y, then z, then same for right hand
    # axis=1 concatenates along the feature dimension, not the frame dimension
    seq = np.concatenate([ # change it so that each frame is represented by a 2D vector of all the landmarks in order
        left[:, :, 0],   # left hand x: indices 0-20
        left[:, :, 1],   # left hand y: indices 21-41
        left[:, :, 2],   # left hand z: indices 42-62
        right[:, :, 0],  # right hand x: indices 63-83
        right[:, :, 1],  # right hand y: indices 84-104
        right[:, :, 2],  # right hand z: indices 105-125
    ], axis=1)           # final shape: (n_frames, 126)


    # Replace NaN values with 0
    # NaN occurs when MediaPipe could not detect a hand in a frame
    # I decided to use 0 instead of imputation because: zeros carry meaning that the hand is not present.
    # MediaPipe is all-or-nothing, so either all 21 landmarks or none will be present; the models can learn that zeros = one-handed sign
    seq = np.nan_to_num(seq, nan=0.0) # replaces all the NaN values with 0.0
    
    # identify and remove frames where BOTH hands are completely absent
    left_missing  = np.all(seq[:, 0:63]   == 0, axis=1)  # True if left hand all zeros
    right_missing = np.all(seq[:, 63:126] == 0, axis=1)  # True if right hand all zeros
    valid_frames  = ~(left_missing & right_missing)      # keep if at least one hand present
    seq = seq[valid_frames]
    
    # skip this clip if too few valid frames remain after filtering, removing clips where MediaPipe almost completely failed
    if len(seq) < MIN_FRAMES:
        return None
    
    # return the cleaned sequence
    # shape: (T, 126) where T >= MIN_FRAMES and T varies per clip
    # T will be standardized to TARGET_FRAMES=30 in feature engineering
    return seq

# ── Check on Sample Clip ───────────────────────────────────────────────
# test load_parquet on the first clip in our dataset before running on all 18907
seq = load_parquet(sample_path)
print(f"Shape: {seq.shape}")
print(f"NaN count: {np.isnan(seq).sum()}")
print(f"All zero frames: {np.all(seq == 0, axis=1).sum()}")
print(f"Min: {seq.min():.4f}, Max: {seq.max():.4f}")

In [ ]:
# Main Data Cleaning Loop 
# lists to store cleaned sequences and their labels, "raw" because there is no feature engineering or any changes to the data yet
X_raw, y_raw = [], []

# counter to track when files that are two short get skipped
skipped = 0 

for i, row in train.iterrows(): # the train.csv holds the path to each parquet for each sign 
    
    # build full path to this clip's parquet file, using the path from the train.csv
    path = os.path.join(DATA_DIR, row['path']) 
    
    # load and clean the parquet file using our load_parquet function
    # returns (T, 126) array or None if clip is unusable
    seq = load_parquet(path)
    
    # skip if clip had too few valid hand frames
    if seq is None:
        skipped += 1
        continue
    
    # store the cleaned sequence and its sign label
    X_raw.append(seq)
    y_raw.append(row['sign'])
    
    # print progress every 2000 clips
    if (i + 1) % 2000 == 0:
        print(f"  Processed {i+1}/{len(train)} — loaded {len(X_raw)}, skipped {skipped}")

# ── Cleaning Summary ──────────────────────────────────────────────────────────
print(f"\n=== Data Cleaning Summary ===")
print(f"Total clips:            {len(train)}")
print(f"Successfully loaded:    {len(X_raw)}")
print(f"Total skipped:          {skipped}")
print(f"Retention rate:         {len(X_raw)/len(train)*100:.2f}%")
print(f"\n=== Samples Per Sign After Cleaning ===")
sign_counts = pd.Series(y_raw).value_counts() # counts how many times each sign appears 
print(sign_counts.to_string())
print(f"\nMin samples per sign:  {sign_counts.min()}")
print(f"Max samples per sign:  {sign_counts.max()}")
print(f"Mean samples per sign: {sign_counts.mean():.2f}")
print(f"\nAny sign with 0 samples: {(sign_counts == 0).any()}")

After cleaning the data we maintain 99.7 percent of the data, which is great.
The mean samples per sign decreases from 378.14 to 376.96, which is a negligable decrease.

In [ ]:
# Convert to numpy arrays 
# X_raw is a list of variable length sequences (T, 126)
# we can't stack them yet since T isn't the same for every clip; we'll convert y_raw to numpy now and handle X after feature engineering and normalization

y_raw = np.array(y_raw)

print(f"=== Raw Data Summary ===")
print(f"Total clips loaded: {len(X_raw)}")
print(f"Labels shape: {y_raw.shape}")
print(f"\nSequence length distribution:")
lengths = [len(s) for s in X_raw]
print(f"  Min frames:    {min(lengths)}")
print(f"  Max frames:    {max(lengths)}")
print(f"  Mean frames:   {np.mean(lengths):.1f}")
print(f"  Median frames: {np.median(lengths):.1f}")
print(f"\nFeature vector size: {X_raw[0].shape[1]} features per frame") #The feature vector for each sample's frame

In [ ]:
# Distribution of Frames per Sample
lengths = np.array([len(s) for s in X_raw])

print("=== Frame count distribution ===")
thresholds = [5, 6, 8, 10, 12, 15, 20, 30]
for t in thresholds:
    below = np.sum(lengths < t)
    above = np.sum(lengths >= t)
    print(f"  < {t:3d} frames: {below:6d} ({below/len(lengths)*100:6.2f}%)  |  >= {t:3d} frames: {above:6d} ({above/len(lengths)*100:6.2f}%)")

**Frame Distrubution:**
Based on the distribution of the frames per sign, and the stated 25fps for the clips. I currently think it is best to keep the target frames at 30, as compression would loose information and would most likely cause more harm than stretching the sings with less frames.

The tradeoff was considered and might be revisited based on the results of all the training models.

## Feature Engineering


My current features for this project are the 126 x,y, and z vectors for each frame. And one sample will contain the target frames (either compressed or stretched from the original) to reach the target frames for each sample clip.

Normalization relative to wrist — already doing this, removes position bias

Velocity features — computing how much each landmark moves between frames captures motion patterns, which is important since many signs look similar in static frames but have different movements

For Random Forest specifically — since RF can't learn temporal patterns, we need to engineer temporal features manually: mean, std, min, max, displacement of each coordinate across all frames. This is what the professor's comment #5 was asking about.

### Interpolation

We need to stretch and compress the different samples with n amount of frames to adheer to the target frames that we have set.

We are using linear interpolation here as it is faster and and will not overshoot/create landmarks that did not exist before. We are keeping the target frames higher, as described earlier, to prevent loss of meaningful features from compression of the longer sample clips/signs.

Non-linear interpolation, especially for compressing the longer clips, was considered. It would maintain the variance of the data better if we remove the clips similar to those before and after it. Ultimiatly this would create variation of inconsistent timesteps in the data which could confuse some of the modles, like LSTM. Addiitionally, the inference predictions would need to be more complicated and interpolation as a whole would be more complicated.

In [ ]:
# Interpolation 
# Resamples every sequence to exactly TARGET_FRAMES=30 frames
# using linear interpolation along the time axis, adding or compressing the amount of frames per sample

# Short clips get stretched to 30 — frames get duplicated/interpolated
# Long clips get compressed to 30 — frames get subsampled

# interpolate function for each sample clip
def interpolate(seq, target=TARGET_FRAMES):
    T = len(seq)
    
    # if already correct length, return as is
    if T == target:
        return seq
    
    # create evenly spaced points from 0 to 1 for original and target lengths
    # think of these as timestamps: original clip has T timestamps, we want 30
    x_old = np.linspace(0, 1, T)       # T points between 0 and 1
    x_new = np.linspace(0, 1, target)  # 30 points between 0 and 1

    # checks to make sure x_old is increasing, as it is a requirment of the np.interp, should always be correct anyway
    if not np.all(np.diff(x_old) > 0): return False
    
    # create output array of shape (target, 126)
    output = np.zeros((target, seq.shape[1]), dtype=np.float32)
    
    # interpolate each of the 126 features independently across the clip using np.interp
    for i in range(seq.shape[1]):
        output[:, i] = np.interp(x_new, x_old, seq[:, i])
    
    return output # result shape: (target, 126)
    

# ── Test on sample clip ───────────────────────────────────────────────────────
seq_interp = interpolate(seq) # seq represents a single clip, and used outside functions is the sample loop we previously loaded
print(f"Before interpolation: {seq.shape}")
print(f"After interpolation:  {seq_interp.shape}")
print(f"NaN count after interpolation: {np.isnan(seq_interp).sum()}")

Used the numpy np.interp function to interpolate the frames for each sign (https://numpy.org/doc/stable/reference/generated/numpy.interp.html). This function works to fill in missing values between known data points using straight lines. The target data points are redfined using the "x_new = np.linspace(0,1,target)".

We are running interpolation and normalization over the data in one loop.


### Normalization

We need to do normalization based on the description of the data set. 

Data inconsistencies:
- Different Recording Positions: The signers had the recordings of the signing actions taken from different heights and angles. Wrist normalization is important in the way CNNs use poolling and convolution to reduce dependancy on position.
- Different Hand Sizes: The different hand sizes means the wrist at different positions means different things based on the signer. Wrist normalization should center this regardless of hand size.
- Real-time inference: During inference on real time data the wrist and angles will be at different positions than in the training data samples. Without normilization, the model will get confused, having never seen the hand(s) on the screen in the given shape and position.
- Left vs. Right Hand Signers: Some signers use different hands as their main hand sign. Normalization should help midigate the difference for this.

Normalization Approaches
- Original Idea for Normalization:  will be done by subtracting the wrist positon from each landmark coordinate (for each axis) and storing the result as a replacement of the original position of that landmark. This will be done for both hands, given that the hand appears in the frame. The problem with this is that it sets both wrists to zero and loses the spatial relationship between the two hands that is important for signs using two hands.
- Including both normalized and non-normalized data: This would double the feature size and create a lot of dependent redundancy, which is not the best option for this application.
- feature engineering using the original idea and adding a feature for hand distance separation: this is a better option, but still loses a lot of the spatial relationships between the hand landmarks.
- Normalization of the hands relative to a single reference - this is the normalization choice we chose as it keeps the spatial relationship while normalizing the wrist data. For this method we will make the reference point the middle point between the two wrists, normal normalization happens with just one hand.

We will use the afformentioned Normalization around a single reference point, which is the mid point between the two wrist coordinates.


In [ ]:
# Wrist Normalization 
# Normalizes both hands relative to a single reference point: the midpoint between the two wrists. 
# This preserves the spatial relationship between hands while still achieving position invariance.

# Edge cases:
# - Only left hand detected:  midpoint = left wrist  (standard normalization)
# - Only right hand detected:  midpoint = right wrist (standard normalization)
# - Neither hand detected:  skip normalization entirely
# - Both hands detected: midpoint = average of both wrists

# normalize each sample clip individually with this normalize function
def normalize(seq):
    seq = seq.copy()  # don't modify the original array
    
    for frame in range(len(seq)): # for each frame in the clip 
        
        # get left wrist position (index 0=x, 21=y, 42=z)
        lx, ly, lz = seq[frame, 0], seq[frame, 21], seq[frame, 42]
        
        # get right wrist position (index 63=x, 84=y, 105=z)
        rx, ry, rz = seq[frame, 63], seq[frame, 84], seq[frame, 105]
        
        # determine which hands are detected in this frame
        left_detected  = not (lx == 0 and ly == 0 and lz == 0)
        right_detected = not (rx == 0 and ry == 0 and rz == 0)
        
        # skip this frame entirely if neither hand is detected
        if not left_detected and not right_detected:
            continue
        
        # compute reference point based on which hands are present
        if left_detected and right_detected:
            # both hands detected — use midpoint between wrists
            # this preserves the spatial relationship between hands
            ref_x = (lx + rx) / 2
            ref_y = (ly + ry) / 2
            ref_z = (lz + rz) / 2
        elif left_detected:
            # only left hand — use left wrist as reference
            ref_x, ref_y, ref_z = lx, ly, lz
        else:
            # only right hand — use right wrist as reference
            ref_x, ref_y, ref_z = rx, ry, rz
        
        # subtract reference point from ALL hand landmarks, preserving their spatial relationship
        seq[frame, 0:21]    -= ref_x  # left hand x coords
        seq[frame, 21:42]   -= ref_y  # left hand y coords
        seq[frame, 42:63]   -= ref_z  # left hand z coords
        seq[frame, 63:84]   -= ref_x  # right hand x coords
        seq[frame, 84:105]  -= ref_y  # right hand y coords
        seq[frame, 105:126] -= ref_z  # right hand z coords
    
    return seq

# ── Test on sample clip ───────────────────────────────────────────────────────
seq_norm = normalize(seq_interp)
print(f"Shape after normalization: {seq_norm.shape}")
print(f"NaN count: {np.isnan(seq_norm).sum()}")

# verify left wrist is at correct position after normalization
print(f"\nLeft wrist x (should be -half the wrist separation or 0 if one handed):")
print(seq_norm[:, 0].round(4))
print(f"\nRight wrist x (should be +half the wrist separation or 0 if not detected):")
print(seq_norm[:, 63].round(4))

In [ ]:
# Apply Feature Engineering to All Clips
X_processed = []

for i, seq in enumerate(X_raw):
    seq = interpolate(seq)   # resample to 30 frames
    seq = normalize(seq)     # center relative to wrist midpoint
    X_processed.append(seq)
    
    if (i + 1) % 2000 == 0:
        print(f"  Processed {i+1}/{len(X_raw)}")

# stack into one numpy array
X = np.array(X_processed, dtype=np.float32)
y = y_raw.copy()

print(f"\nFinal dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"NaN count: {np.isnan(X).sum()}")

In [ ]:
from sklearn.model_selection import train_test_split

# ── Stratified Train/Val/Test Split (70/15/15) ────────────────────────────────
# Stratified means each sign is proportionally represented in all three sets
# We split before saving so test set is never seen during training or tuning

# first split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_SEED
)

# second split: split temp 50/50 into val and test (15% each)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_SEED)

print(f"Train: {X_train.shape} — {len(X_train)} samples")
print(f"Val:   {X_val.shape} — {len(X_val)} samples")
print(f"Test:  {X_test.shape} — {len(X_test)} samples")
print(f"\nSamples per sign in test set:")
print(pd.Series(y_test).value_counts().to_string())